# 01 — Cohort and paired-valid target audit

This notebook decides which pose sequences are suitable for the later
analysis and checks that the left-versus-right motion measurement behaves
as intended. A **pose sequence** is a series of video frames represented by
estimated body-landmark coordinates instead of by the original images. A
**cohort** is simply the collection of sequences retained for analysis.

The data come from the Gait Abnormality in Video Dataset (GAVD). Although
that is the dataset's published name, this project treats its folder labels
only as dataset annotations. This notebook does not diagnose anyone and
does not validate a clinical measurement.

In the `smoke` profile, the poses are generated test data. Smoke mode checks
that the software works from beginning to end; its output is not scientific
evidence. In the `paper` profile, the same checks run on the empirical data.

## What this notebook is checking

This notebook turns the available pose files into the **locked analysis
cohort** used by later notebooks. "Locked" means that the acceptance rules
were written down before examining the later model results. Keeping those
rules fixed helps prevent a result from being improved by quietly changing
which sequences are included.

Read the notebook as a sequence of six questions:

1. Which protocol and data profile are active?
2. Which pose sequences pass the pre-specified quality-control (QC) rules?
3. Can the left-versus-right motion target be computed from observed
   coordinates alone?
4. Does anatomical mirroring reverse the target exactly and undo itself
   when applied twice?
5. Were the audited cohort and its history saved for the later stages?
6. What do the final chart and audit numbers mean?

This is a **data and implementation audit**, not a model-performance result.
No machine-learning model is trained and no diagnosis is predicted here.

## Plain-language glossary

Here are the main terms used below:

- **Quality control (QC)** means applying the pre-written checks that decide
  whether a pose sequence contains enough usable information.
- A **body landmark** is an estimated point such as a shoulder, knee, ankle,
  heel, or foot point. Each point has three coordinates, called `x`, `y`,
  and `z`.
- A **frame transition** is the movement from one video frame to the next
  available frame.
- **Paired-valid** means that a left landmark and its matching right
  landmark are both visible at the start and end of the same transition.
  Comparing the two sides on exactly the same transitions avoids giving one
  side an unfair advantage because it was visible more often.
- The **target** is the single number that later models will try to predict.
  Here it summarizes relative left-versus-right motion. It is derived from
  coordinates and is not a clinical outcome.
- **Interpolation** means filling a short gap by estimating values between
  two observed points. Interpolation may make model input easier to use, but
  interpolated values are not allowed to define the target.
- A **sentinel** is a placeholder stored where a coordinate is invalid. The
  validity mask tells the program to ignore it, whatever its numeric value.
- In this notebook, **authorized landmarks** are simply the body landmarks
  selected in the frozen protocol for model input. "Authorized" here does
  not mean that an ethics or data-release review has been approved.
- A **patch** is one four-frame block used by the model. A complete patch has
  valid information throughout that block.
- A **finite** target is an ordinary usable number, rather than a missing or
  undefined value.
- **Provenance** means the recorded history of the data, including which
  files, pose model, and extraction version produced them.
- A **digest** is a long content fingerprint. SHA stands for Secure Hash
  Algorithm; **SHA-256** produces a 256-bit fingerprint. If relevant content
  changes, its digest changes, allowing later notebooks to detect a mismatch.

## Step 1 — Confirm the run context

The next cell locates the experiment suite, loads the frozen protocol, and
prints four useful identifiers:

- `suite` is the folder containing this experiment;
- `profile` says whether this is synthetic `smoke` data or the empirical
  `paper` run;
- `artifacts` is the folder where this notebook writes its derived files;
- `protocol` shows the first characters of the protocol digest. The full
  digest is a fingerprint of the rules and settings. Later files must carry
  the same fingerprint, which prevents different protocol versions from
  being mixed accidentally.

Check this line before interpreting anything below. A smoke-profile figure
tests the plumbing only and is not scientific evidence.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

## Steps 2–5 — Build and verify the cohort

The next cell performs the substantive audit in a fixed order:

**Step 2: inventory and prepare the pose files.** The program first checks
that the file counts and fingerprints match the frozen inventory. It then
converts each sequence into 64 time steps with 33 body landmarks. Short gaps
may be interpolated for model input, and a separate validity mask records
which coordinates the model may use. The untouched observed-coordinate path
is kept separate for calculating the target.

**Step 3: apply locked quality control.** The `prepare_cohort` function
checks whether a sequence contains enough usable landmark coverage, enough
complete four-frame blocks, and enough information to calculate the target.
A sequence either enters the cohort or receives an explicit exclusion
reason. Importantly, acceptance does not depend on whether the target is
positive, negative, large, or small.

**Step 4: calculate and independently reconstruct the target.** Five matching
left/right landmark pairs are used: shoulders, knees, ankles, heels, and
foot-index points. For each pair, the program:

1. keeps only transitions where both landmarks are observed at both ends;
2. calculates how fast each side moved;
3. takes the median, or middle, speed for each side; and
4. computes `(left speed - right speed) / (left speed + right speed)`.

The five pair values are averaged to make one target. A simple example is a
left speed of 3 and right speed of 2, which gives `(3 - 2) / (3 + 2) = 0.2`.
Positive values indicate more left-side motion under this definition;
negative values indicate more right-side motion. The code recalculates this
value from the saved pair contrasts and checks that the answer matches to
within a tiny floating-point tolerance. "Floating point" is the computer's
approximate way of storing decimal numbers.

**Step 5: test the mirror rules.** An anatomical mirror flips the horizontal
coordinate and swaps each named left landmark with its right partner. That
operation must reverse the sign of the target. Applying it twice must return
the original coordinates and validity mask. This mirror-twice property is
sometimes called an **involution**. The program also replaces invalid
coordinates with enormous sentinel numbers and confirms that the target does
not change. If any assertion fails, execution stops because later results
would not have a trustworthy left-versus-right interpretation.

Finally, the accepted arrays, manifest, and metadata are saved with a cohort
digest. A **manifest** is a table listing the retained sequences. **Metadata**
is information describing the data rather than the pose values themselves.
Later notebooks use the digest to prove that they loaded this exact handoff.

## The paired-valid target in one picture

The same visibility rule is applied to both sides before speeds are compared.
For example, if the left knee is visible across 20 transitions but the right
knee is visible across only 12 of those, the calculation does not compare 20
left transitions with 12 right transitions. It keeps only transitions jointly
visible for both knees at both endpoints. This is what **paired-valid** means.

<svg viewBox="0 0 1050 285" width="100%" role="img"
     aria-labelledby="target-flow-title target-flow-description"
     xmlns="http://www.w3.org/2000/svg">
  <title id="target-flow-title">Construction and mirror audit of the paired-valid target</title>
  <desc id="target-flow-description">Observed coordinates and validity masks
  select jointly visible transitions. Median left and right speeds form a
  normalized contrast for each of five landmark pairs, which are averaged into
  the target. Anatomical mirroring must negate that target.</desc>
  <defs><marker id="arrow01" markerWidth="8" markerHeight="8" refX="7"
    refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#475569"/></marker></defs>
  <style>
    .box01{fill:#f8fafc;stroke:#334155;stroke-width:1.5}
    .valid01{fill:#ecfdf5;stroke:#047857;stroke-width:1.5}
    .target01{fill:#eff6ff;stroke:#2563eb;stroke-width:1.8}
    .mirror01{fill:#fff7ed;stroke:#c2410c;stroke-width:1.5}
    .line01{stroke:#475569;stroke-width:1.8;fill:none;marker-end:url(#arrow01)}
    .h01{font:600 14px system-ui,sans-serif;fill:#0f172a}
    .s01{font:12px system-ui,sans-serif;fill:#475569}
  </style>
  <rect class="box01" x="15" y="35" width="170" height="78" rx="9"/>
  <text class="h01" x="100" y="64" text-anchor="middle">Observed coordinates</text>
  <text class="s01" x="100" y="85" text-anchor="middle">plus validity masks</text>
  <text class="s01" x="100" y="102" text-anchor="middle">no interpolated target values</text>
  <rect class="valid01" x="235" y="35" width="190" height="78" rx="9"/>
  <text class="h01" x="330" y="64" text-anchor="middle">Paired-valid transitions</text>
  <text class="s01" x="330" y="85" text-anchor="middle">both sides visible</text>
  <text class="s01" x="330" y="102" text-anchor="middle">at both endpoints</text>
  <rect class="box01" x="475" y="20" width="220" height="108" rx="9"/>
  <text class="h01" x="585" y="49" text-anchor="middle">One landmark pair</text>
  <text class="s01" x="585" y="70" text-anchor="middle">median left speed L</text>
  <text class="s01" x="585" y="88" text-anchor="middle">median right speed R</text>
  <text class="h01" x="585" y="112" text-anchor="middle">contrast = (L − R) / (L + R)</text>
  <rect class="target01" x="745" y="35" width="285" height="78" rx="9"/>
  <text class="h01" x="887" y="64" text-anchor="middle">Coordinate-derived target</text>
  <text class="s01" x="887" y="85" text-anchor="middle">mean across five registered pairs</text>
  <text class="s01" x="887" y="102" text-anchor="middle">shoulders, knees, ankles, heels, foot-index</text>
  <path class="line01" d="M185 74 L235 74"/><path class="line01" d="M425 74 L475 74"/><path class="line01" d="M695 74 L745 74"/>
  <rect class="mirror01" x="235" y="185" width="460" height="72" rx="9"/>
  <text class="h01" x="465" y="213" text-anchor="middle">Anatomical mirror audit</text>
  <text class="s01" x="465" y="235" text-anchor="middle">flip horizontal coordinate + swap left/right landmarks</text>
  <rect class="target01" x="745" y="185" width="285" height="72" rx="9"/>
  <text class="h01" x="887" y="213" text-anchor="middle">Required result</text>
  <text class="h01" x="887" y="238" text-anchor="middle">target(mirror(x)) = −target(x)</text>
  <path class="line01" d="M330 113 L330 185"/><path class="line01" d="M695 221 L745 221"/>
</svg>

A contrast is bounded between -1 and +1 when speeds are nonnegative and their
sum is positive. A value of
+0.2 means the median left speed exceeded the right speed under this formula; it
does not mean “20% impairment.” Mirroring swaps $L$ and $R$, so the numerator
changes sign while the denominator stays the same. That algebra explains why
exact sign reversal is the required implementation check.

## How to read the cohort-audit progress display

The five stages cover pose inventory and quality control, independent target
reconstruction, mirror-twice checks, artifact saving, and figure rendering. The
mirror stage also reports how many accepted sequences have been checked, so a
large empirical cohort does not look frozen.

ETA is based on completed stage durations, but cohort preparation is usually much
more expensive than plotting or saving. The estimate can therefore move sharply
after the first stage. Progress describes completed computation only; the final
attrition counts and assertion results determine whether the cohort is valid.

In [ ]:
from collections import Counter

import numpy as np

from laterality.data import prepare_cohort, save_cohort
from laterality.geometry import anatomical_mirror
from laterality.visualization import cohort_figure
from notebook_progress import NotebookTaskProgress

cohort_progress = NotebookTaskProgress(
    "Cohort and target audit progress",
    "stage",
    refresh_seconds=0.25,
)
cohort_progress.start(5, profile=context.profile)

with cohort_progress.unit(1, "Inventory, prepare, and quality-control pose sequences"):
    cohort = prepare_cohort(context)

with cohort_progress.unit(2, "Reconstruct targets and verify target contracts"):
    reconstructed_targets = np.asarray(
        [
            np.mean(row[np.isfinite(row)])
            for row in cohort.pair_contrasts
        ],
        dtype=np.float64,
    )
    assert np.allclose(
        reconstructed_targets,
        cohort.table["target"].to_numpy(dtype=np.float64),
        rtol=0.0,
        atol=1e-12,
    )

    target_contract = cohort.attrition["target_contract"]
    assert target_contract["checked_finite_targets"] >= len(cohort.table)
    assert target_contract["maximum_mirror_antisymmetry_error"] <= 1e-10
    assert target_contract["maximum_invalid_sentinel_error"] <= 1e-12

checked_involutions = 0
with cohort_progress.unit(
    3,
    "Verify mirror-twice restoration for every accepted sequence",
    total_steps=len(cohort.table),
):
    for checked_involutions, (xyz, valid) in enumerate(
        zip(cohort.model_xyz, cohort.model_valid),
        start=1,
    ):
        mirrored_xyz, mirrored_valid = anatomical_mirror(xyz, valid)
        restored_xyz, restored_valid = anatomical_mirror(
            mirrored_xyz, mirrored_valid
        )
        assert np.array_equal(restored_xyz, xyz)
        assert np.array_equal(restored_valid, valid)
        cohort_progress.update_unit(completed_steps=checked_involutions)

with cohort_progress.unit(4, "Save the verified cohort handoff and summarize exclusions"):
    artifact_paths = save_cohort(context, cohort)
    exclusion_reason_counts = Counter(
        item["reason"] for item in cohort.attrition["exclusions"]
    )
    attrition_summary = {
        key: value
        for key, value in cohort.attrition.items()
        if key != "exclusions"
    }
    attrition_summary["exclusion_reason_counts"] = dict(
        sorted(exclusion_reason_counts.items())
    )

with cohort_progress.unit(5, "Render the attrition and target-distribution figure"):
    show_inline(cohort_figure(context, cohort))
cohort_progress.complete(status="Cohort and target audit complete")
{
    "attrition": attrition_summary,
    "cohort_digest": cohort.cohort_digest,
    "target_contract": target_contract,
    "checked_model_lane_mirror_involutions": checked_involutions,
    "artifacts": {key: str(value) for key, value in artifact_paths.items()},
}

## Step 6 — Read the figure and audit record

**Start with the left panel.** "Attrition" means the reduction from the
starting set to the usable set. `Input poses` is the number of available pose
sequences presented to quality control. `QC eligible` is the subset retained
for analysis, and `Excluded` is the remainder. In the displayed paper-profile
run, 625 of 642 sequences are retained and 17 are excluded. Thus about 97.4%
are retained and 2.6% are excluded. The excluded bar is not a model error
rate and does not label people as good or bad data. It records sequences that
could not satisfy the fixed coverage or target-computability rules.

**Then read the right panel.** This is a histogram, which groups numeric
values into ranges and uses bar height to show how many sequences fall in
each range. The horizontal axis is the coordinate-derived motion contrast,
and the vertical axis is the number of retained sequences. The black line
marks zero. Values to its right indicate relatively more left-side motion;
values to its left indicate relatively more right-side motion. Values near
zero indicate similar motion on the two sides under this specific formula.
The presence of values on both sides confirms that the target keeps a sign.
It does not establish that the sample is clinically symmetric, unbiased, or
representative of a population. This chart is not a diagnosis, clinical
scale, class label, or model-performance score.

**Finally, read the printed audit record below the plot.** The main fields
mean the following:

- `input_sequences`, `accepted_sequences`, and `excluded_sequences` repeat
  the counts shown in the left panel;
- `accepted_sources` counts source videos rather than pose sequences. One
  source video can produce more than one sequence, so this number is smaller;
- `inventory` contains the frozen file counts and provenance fingerprints;
- `exclusion_reason_counts` explains why sequences failed quality control.
  A semicolon means that a sequence failed more than one check;
- `target_contract` reports the mirror and invalid-sentinel test errors.
  Zero is ideal, and extremely small values can arise from decimal rounding;
- `checked_model_lane_mirror_involutions` counts how many processed model
  inputs passed the mirror-twice check;
- `cohort_digest` is the content fingerprint used for later history checks;
- `artifacts` lists the saved handoff files. The compressed NumPy file
  (`.npz`) stores arrays, the comma-separated values file (`.csv`) stores the
  manifest table, and the JavaScript Object Notation file (`.json`) stores
  metadata.

The inventory gives useful context for the first bar. The run began with
666 annotation files. Of these, 642 had a matching pose archive, meaning a
saved bundle of extracted body coordinates; 24 did not. That is why the
chart begins at 642 rather than 666. The annotations refer to 103 source
videos, and the final cohort contains usable sequences from 93 of them.
`extraction_version_counts` shows which software version produced each pose
archive, while `pose_model` names the body-landmark detector. These values
are provenance checks, not measures of scientific performance.

The exclusion labels can contain `target_not_computable`, meaning the five
left/right pairs did not all have enough shared visible transitions;
`insufficient_authorized_coverage`, meaning too little of the selected model
input was valid; or `insufficient_authorized_patches`, meaning too few
complete four-frame blocks remained.

In this run, `checked_finite_targets` can be larger than the 625 accepted
sequences. That is expected: the target checks run before all coverage rules,
so a sequence can have a valid target and still be excluded for insufficient
model-input coverage.

The key takeaway is modest: the cohort is internally consistent with the
frozen quality-control and mirror rules. This figure alone says nothing about whether a
model learns laterality or generalizes to new videos or people.

## Specific interpretation of the current paper-profile findings

The complete inventory forms a useful funnel. There are 666 annotation files,
642 matching pose archives, and therefore 24 annotations without a usable pose
archive at this stage. Locked quality control then retains 625 of the 642 pose
sequences and excludes 17. The retention rate is 97.35%. The accepted sequences
come from 93 source videos, while the official inventory contains 103 source
videos. Sequence and source counts are different because one source video can
yield multiple extracted sequences.

The 17 exclusion records break down into four exact combinations:

| Recorded reason combination | Sequences |
|---|---:|
| target not computable + insufficient authorized coverage | 8 |
| target not computable + insufficient coverage + insufficient complete patches | 4 |
| insufficient authorized coverage only | 3 |
| target not computable only | 2 |

These rows describe failed technical requirements, not participants or clinical
categories. Because one row can fail several checks, the semicolon-separated
combination should be read as one sequence's complete reason record. The target
was finite for 628 sequences even though only 625 entered the cohort. This is
expected: three sequences could form a target but failed a later model-input
coverage rule.

The accepted target distribution spans approximately -0.195 to +0.215. Its mean
is -0.0061, median is -0.0050, and middle 50% runs from about -0.0434 to +0.0286.
There are 336 negative and 289 positive sequence values, with no exact zeros.
This tells us that the computed quantity covers both signs and is centered near
zero at the sequence level. It does not show clinical balance, population
representativeness, or absence of confounding. The histogram counts sequences,
so a source producing many sequences contributes more bars; later performance
metrics deliberately give each source equal total weight.

Both recorded algebraic error maxima are exactly 0.0 in this run:
`maximum_mirror_antisymmetry_error` and
`maximum_invalid_sentinel_error`. All 625 accepted model inputs also passed the
mirror-twice restoration check. These are strong implementation checks: the
mirror and validity-mask code obey the frozen mathematical contract on the
audited inputs. They do not validate the target as a medical measurement and do
not show that a learned model will recover it.

The narrow conclusion is: **the empirical cohort handoff is complete and
internally consistent with the locked inventory, quality-control, validity,
target-reconstruction, and mirror rules.** The next notebook may split these 93
sources. No model-performance claim begins until the held-out evaluation.